# Browse every spaCR workflow

Choose a workflow, configure inputs, then explicitly enable Run.
[Guided notebooks](README.md) · [Python guide](https://einarolafsson.github.io/spacr/python_api.html)

## 1. Choose a workflow
Choose, then rerun configuration after changing workflow.


In [ ]:
from spacr.version import version_str
from spacr.cli import MODULES, resolve_module, resolve_settings
from IPython.display import Markdown, display

print("spaCR", version_str)
module_key = "measure"
module_picker = None
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional picker unavailable; edit module_key above.")
else:
    module_picker = widgets.Dropdown(
        options=[(f"{key}: {row.summary}", key) for key, row in sorted(MODULES.items())],
        value=module_key, description="Workflow:", layout=widgets.Layout(width="100%"))
    display(module_picker)


## 2. Configure the selected workflow
Load saved CSV/JSON settings or edit typed `key=value` overrides.


In [ ]:
module = resolve_module(module_picker.value if module_picker is not None else module_key)
settings_file = None
overrides = []  # Example: ["src=/absolute/path/to/plate", "channels=[0,1]"]
settings = resolve_settings(module, settings_file, overrides)
dotted = module.entry.replace(":", ".")
api_url = "https://einarolafsson.github.io/spacr/api/" + module.module_name.replace(".", "/") + "/index.html#" + dotted
display(Markdown(f"### {module.key}: {module.summary}\n\n[Public API]({api_url})"))
print("Required inputs:", *module.requires, sep="\n- ")
print("Writes:", *module.writes, sep="\n- ")
print(f"{len(settings)} current settings loaded. Edit overrides above or settings below.")


### Advanced settings editor (optional)
Click Advanced settings, edit, then Apply settings. Analysis stays off.


In [ ]:
if module_picker is not None:
    try:
        from _spacr_notebook_widgets import show_workflow_settings_editor
    except ImportError:
        print("Optional editor helper unavailable; edit settings or overrides in Python.")
    else:
        advanced, editor, apply_button = show_workflow_settings_editor(settings, module)
else:
    print("Edit settings directly, or edit overrides and rerun the configuration cell.")


## 3. Run only when ready
Enable only after configuring inputs; analysis may write files or download models.


In [ ]:
RUN_WORKFLOW = False
if RUN_WORKFLOW:
    from importlib import import_module
    function = getattr(import_module(module.module_name), module.func_name)
    result = function(settings if module.call_style == "settings" else settings["src"])
else:
    print("No analysis started. Configure inputs and enable RUN_WORKFLOW when ready.")
